# 04 — Metric Perturbations (CI‑Aware)
**Abstract.**  
Compute symbolic metric perturbations  
$(\delta g_{ij} = \alpha \cdot \text{Hessian} + \beta \cdot \omega\)$,  
extract scalar modes $((\Phi, \Psi)\)$, and compute a symbolic power spectrum.

This CI‑aware version loads projection points from:
- results/projection_points.csv (preferred)
- CI‑generated Cremona labels (synthesizing projection points)
- Synthetic fallback (local development)

Uses `src.physics.metric_perturbations` when available; otherwise uses safe fallbacks.


In [ ]:
import sys, os
sys.path.append('src')
sys.path.append('../src')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(7)

os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
if os.path.exists('results/projection_points.csv'):
    df = pd.read_csv('results/projection_points.csv')
    points = df[['x','y','z']].values
    print("Loaded results/projection_points.csv")

elif RUNNING_IN_CI and os.path.exists(CI_LABELS):
    print("CI mode: synthesizing projection points from CI labels")
    labels = pd.read_csv(CI_LABELS)
    n = len(labels)
    df = pd.DataFrame({
        'x': np.random.rand(n),
        'y': np.random.rand(n),
        'z': np.random.rand(n)
    })
    points = df[['x','y','z']].values

else:
    print("Projection points not found; generating synthetic points.")
    points = np.random.rand(200,3)


In [ ]:
try:
    from src.physics.metric_perturbations import MetricPerturbations
    MP = MetricPerturbations()

    delta_g_list = [MP.delta_g(p) for p in points]
    scalar_modes = np.array([MP.scalar_modes(p) for p in points])  # (phi, psi)
    traces = np.array([np.trace(dg) for dg in delta_g_list])

    print("Computed delta_g and scalar modes using src.physics.metric_perturbations")

except Exception as e:
    print("MetricPerturbations not available; using fallback approximations.", e)

    delta_g_list = [np.eye(3)*np.sum(p) for p in points]
    scalar_modes = np.vstack([[np.sum(p), -np.sum(p)] for p in points])
    traces = np.array([np.trace(dg) for dg in delta_g_list])


In [ ]:
modes = scalar_modes[:,0] + scalar_modes[:,1]

fft = np.fft.rfft(modes - np.mean(modes))
ps = np.abs(fft)**2
freqs = np.fft.rfftfreq(len(modes), d=1.0)

plt.figure(figsize=(6,4))
plt.loglog(freqs[1:], ps[1:], '-o', markersize=3)
plt.xlabel('k (arb)')
plt.ylabel('Power')
plt.title('Symbolic Power Spectrum')
plt.savefig('results/symbolic_power_spectrum.png', dpi=200)
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(traces, bins=40, color='C3', alpha=0.8)
plt.title('Distribution of trace(δg)')
plt.xlabel('trace')
plt.savefig('results/delta_g_trace_hist.png')
plt.show()

pd.DataFrame({'trace_delta_g': traces}).to_csv('results/delta_g_summary.csv', index=False)
print("Saved results/delta_g_summary.csv and figures.")


**Interpretation.**  
The symbolic power spectrum and δg traces provide a compact summary of metric perturbations induced by entropy curvature.  
Next: compute $H_{\mathrm{eff}}$ and fit to local/CMB proxies (Notebook 05).


In [ ]:
print("Notebook: 04_metric_perturbations")
print("Data source:",
      "projection_points.csv" if os.path.exists('results/projection_points.csv')
      else "CI labels" if RUNNING_IN_CI else "synthetic")
print("Outputs: results/delta_g_summary.csv, results/symbolic_power_spectrum.png")
